In [3]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [4]:
# Cell 2 — Load & Validate Dataset
df = pd.read_csv(
    r'C:\Users\Gurveer\project-18-love-at-first-agent\data\raw\Speed Dating Data.csv',
    encoding='ISO-8859-1'
)

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['match'].value_counts(normalize=True).round(4))
print(f"\nMatch rate: {df['match'].mean()*100:.1f}%")
print(f"\nFirst 20 columns:")
print(list(df.columns[:20]))
df.head(3)

Dataset shape: (8378, 195)

Target distribution:
match
0    0.8353
1    0.1647
Name: proportion, dtype: float64

Match rate: 16.5%

First 20 columns:
['iid', 'id', 'gender', 'idg', 'condtn', 'wave', 'round', 'position', 'positin1', 'order', 'partner', 'pid', 'match', 'int_corr', 'samerace', 'age_o', 'race_o', 'pf_o_att', 'pf_o_sin', 'pf_o_int']


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


In [5]:
# Cell 3 — Data Cleaning & Leakage Removal
# Remove data leakage columns (decision columns reveal the outcome)
leakage_cols = ['decision', 'decision_o', 'match']
id_cols = ['iid', 'id', 'idg', 'condtn', 'wave', 'round', 
           'position', 'positin1', 'order', 'partner', 'pid']

# Target
y = df['match'].copy()

# Drop leakage and ID columns
df_clean = df.drop(columns=leakage_cols + id_cols, errors='ignore')

# Keep only numeric columns
df_clean = df_clean.select_dtypes(include=[np.number])

# Impute missing values with median
df_clean = df_clean.fillna(df_clean.median())

print(f"Shape after cleaning:        {df_clean.shape}")
print(f"Missing values remaining:    {df_clean.isnull().sum().sum()}")
print(f"Leakage columns removed:     {leakage_cols}")
print(f"ID columns removed:          {len(id_cols)}")
print(f"Features remaining:          {df_clean.shape[1]}")

# Save cleaned data
import os
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-18-love-at-first-agent\outputs'
os.makedirs(output_dir, exist_ok=True)
df_clean['match'] = y
df_clean.to_csv(f'{output_dir}\\cleaned_data.csv', index=False)
df_clean = df_clean.drop(columns=['match'])
print(f"\nCleaned data saved to outputs/")

Shape after cleaning:        (8378, 175)
Missing values remaining:    0
Leakage columns removed:     ['decision', 'decision_o', 'match']
ID columns removed:          11
Features remaining:          175

Cleaned data saved to outputs/


In [6]:
# Cell 4 — Feature Selection with RFE
X = df_clean.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# RFE with Random Forest to select top 10 features
print("Running Recursive Feature Elimination...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=10, step=5)
rfe.fit(X_train, y_train)

top_features = X.columns[rfe.support_].tolist()
print(f"\nTop 10 features selected by RFE:")
for i, f in enumerate(top_features, 1):
    print(f"  {i:02d}. {f}")

# Save top features
import json
with open(f'{output_dir}\\top_features.json', 'w') as f:
    json.dump(top_features, f, indent=2)
print(f"\nFeatures saved to outputs/top_features.json")

Running Recursive Feature Elimination...

Top 10 features selected by RFE:
  01. dec_o
  02. attr_o
  03. fun_o
  04. shar_o
  05. like_o
  06. dec
  07. attr
  08. fun
  09. like
  10. prob

Features saved to outputs/top_features.json


In [7]:
# Cell 5 — Remove Remaining Leakage & Retrain RFE
# dec and dec_o are per-date decisions — they ARE leakage
# Remove them and rerun RFE for clean features
leakage_extra = ['dec', 'dec_o']
df_clean2 = df_clean.drop(columns=leakage_extra, errors='ignore')

X2 = df_clean2.copy()
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

print("Rerunning RFE without decision leakage columns...")
rf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe2 = RFE(estimator=rf2, n_features_to_select=10, step=5)
rfe2.fit(X2_train, y2_train)

top_features_clean = X2.columns[rfe2.support_].tolist()
print(f"\nTop 10 CLEAN features (no leakage):")
for i, f in enumerate(top_features_clean, 1):
    print(f"  {i:02d}. {f}")

with open(f'{output_dir}\\top_features_clean.json', 'w') as f:
    json.dump(top_features_clean, f, indent=2)
print(f"\nClean features saved.")

Rerunning RFE without decision leakage columns...

Top 10 CLEAN features (no leakage):
  01. int_corr
  02. pf_o_att
  03. attr_o
  04. like_o
  05. prob_o
  06. attr1_1
  07. attr2_1
  08. attr
  09. like
  10. prob

Clean features saved.


In [8]:
# Cell 6 — Train XGBoost on Clean Features
X_train_clean = X2_train[top_features_clean]
X_test_clean  = X2_test[top_features_clean]

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train_clean, y2_train)
xgb_proba = xgb_model.predict_proba(X_test_clean)[:, 1]
auc = roc_auc_score(y2_test, xgb_proba)

print(f"=== XGBoost Results ===")
print(f"ROC AUC Score: {auc:.4f}")
print(f"Features used: {top_features_clean}")

=== XGBoost Results ===
ROC AUC Score: 0.8553
Features used: ['int_corr', 'pf_o_att', 'attr_o', 'like_o', 'prob_o', 'attr1_1', 'attr2_1', 'attr', 'like', 'prob']


In [9]:
# Cell 7 — ROC Curve Visualization
fpr, tpr, _ = roc_curve(y2_test, xgb_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'XGBoost (AUC={auc:.4f})',
    line=dict(color='crimson', width=2.5)
))
fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Random Classifier'
))
fig.update_layout(
    title='ROC Curve — Love at First Agent (XGBoost)',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_dark',
    width=750, height=480
)
fig.show()
print(f"ROC AUC: {auc:.4f}")

ROC AUC: 0.8553


In [10]:
# Cell 8 — Define LangGraph Agent Tools
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langgraph.graph import add_messages
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

@tool
def profile_data(info: str) -> str:
    """Profile the speed dating dataset and return key statistics."""
    return json.dumps({
        "total_rows":    int(df.shape[0]),
        "total_columns": int(df.shape[1]),
        "match_rate":    f"{float(y.mean()*100):.1f}%",
        "class_balance": {
            "no_match": f"{float((y==0).mean()*100):.1f}%",
            "match":    f"{float((y==1).mean()*100):.1f}%"
        },
        "recommendation": "Use ROC AUC as primary metric due to class imbalance"
    })

@tool
def check_leakage(features: str) -> str:
    """Check a comma-separated list of features for data leakage."""
    leakage = ['decision', 'decision_o', 'match', 'dec', 'dec_o']
    flagged = [f.strip() for f in features.split(',') if f.strip() in leakage]
    safe    = [f.strip() for f in features.split(',') if f.strip() not in leakage]
    return json.dumps({
        "flagged_leakage": flagged,
        "safe_features":   safe,
        "action": f"Remove {flagged} before training" if flagged else "No leakage detected"
    })

@tool
def train_model(feature_list: str) -> str:
    """Train XGBoost on provided features and return ROC AUC."""
    features = [f.strip() for f in feature_list.split(',') 
                if f.strip() in X2.columns]
    if not features:
        return json.dumps({"error": "No valid features provided"})
    
    Xf_train = X2_train[features]
    Xf_test  = X2_test[features]
    
    model = xgb.XGBClassifier(
        n_estimators=100, max_depth=4,
        learning_rate=0.05, random_state=42,
        eval_metric='logloss', verbosity=0
    )
    model.fit(Xf_train, y2_train)
    proba = model.predict_proba(Xf_test)[:, 1]
    auc   = roc_auc_score(y2_test, proba)
    
    return json.dumps({
        "features_used": features,
        "roc_auc":       round(float(auc), 4),
        "verdict":       "Excellent" if auc > 0.85 else "Good" if auc > 0.75 else "Needs improvement"
    })

@tool
def evaluate_match_probability(age: int, attractiveness: float, 
                                fun: float, shared_interests: float) -> str:
    """Predict match probability for a new couple given key attributes."""
    sample = pd.DataFrame([{
        f: 0.0 for f in top_features_clean
    }])
    
    mapping = {
        'age': age, 'attr_o': attractiveness,
        'fun_o': fun, 'shar_o': shared_interests,
        'attr': attractiveness, 'fun': fun
    }
    for col, val in mapping.items():
        if col in sample.columns:
            sample[col] = val
    
    prob = xgb_model.predict_proba(sample[top_features_clean])[0][1]
    return json.dumps({
        "match_probability": f"{prob*100:.1f}%",
        "verdict": "High chance of match!" if prob > 0.5 else "Low match probability"
    })

tools = [profile_data, check_leakage, train_model, evaluate_match_probability]
print(f"Tools registered: {[t.name for t in tools]}")

Tools registered: ['profile_data', 'check_leakage', 'train_model', 'evaluate_match_probability']


In [11]:
# Cell 9 — Build & Run LangGraph ReAct Agent
agent = create_react_agent(llm, tools=tools)

print("=== Love at First Agent — LangGraph ReAct ===\n")

queries = [
    "Profile the speed dating dataset and tell me if ROC AUC is the right metric.",
    "Check these features for leakage: attr_o, fun_o, dec, dec_o, like_o, shar_o",
    "Train a model using these features: attr_o, fun_o, like_o, shar_o, prob, age_o",
    "Predict match probability for a 28-year-old with attractiveness=8, fun=9, shared interests=7"
]

for q in queries:
    print(f"\nQ: {q}")
    print("-" * 60)
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    final  = result["messages"][-1].content
    print(f"Agent: {final}")

=== Love at First Agent — LangGraph ReAct ===


Q: Profile the speed dating dataset and tell me if ROC AUC is the right metric.
------------------------------------------------------------
Agent: The speed dating dataset contains a total of 8,378 rows and 195 columns. The match rate is 16.5%, indicating a significant class imbalance, with 83.5% of the instances being "no match" and 16.5% being "match."

Given this class imbalance, it is recommended to use ROC AUC as the primary metric for evaluating model performance. ROC AUC is particularly suitable in such scenarios as it provides a measure of the model's ability to distinguish between the two classes, regardless of their distribution.

Q: Check these features for leakage: attr_o, fun_o, dec, dec_o, like_o, shar_o
------------------------------------------------------------
Agent: The following features were flagged for leakage:

- **dec**
- **dec_o**

These features should be removed before training. The safe features that can be us

In [12]:
# Cell 10 — Export Results
import os

pd.DataFrame({
    'feature': top_features_clean,
    'rank':    range(1, 11)
}).to_csv(f'{output_dir}\\top_features_clean.csv', index=False)

pd.DataFrame({
    'metric': ['ROC AUC (with leakage)', 'ROC AUC (clean)'],
    'value':  [0.8618, round(auc, 4)]
}).to_csv(f'{output_dir}\\model_results.csv', index=False)

print("Exports saved:")
for f in os.listdir(output_dir):
    print(f"  {f}")

Exports saved:
  cleaned_data.csv
  model_results.csv
  top_features.json
  top_features_clean.csv
  top_features_clean.json


In [14]:
# Cell 11 — Save Agent Pipeline as src/agents.py
src_dir = r'C:\Users\Gurveer\ds-portfolio\project-18-love-at-first-agent\src'
os.makedirs(src_dir, exist_ok=True)

agents_code = (
    'from dotenv import load_dotenv\n'
    'import os\n'
    'load_dotenv(r"C:\\\\Users\\\\Gurveer\\\\ds-portfolio\\\\.env")\n'
    'os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")\n\n'
    'import json\n'
    'from langchain_core.tools import tool\n'
    'from langchain_openai import ChatOpenAI\n'
    'from langgraph.prebuilt import create_react_agent\n\n'
    '@tool\n'
    'def profile_data(info: str) -> str:\n'
    '    """Profile the speed dating dataset."""\n'
    '    return json.dumps({"total_rows": 8378, "match_rate": "16.5%",\n'
    '                       "recommendation": "Use ROC AUC due to class imbalance"})\n\n'
    '@tool\n'
    'def check_leakage(features: str) -> str:\n'
    '    """Check features for data leakage."""\n'
    '    leakage = ["decision", "decision_o", "match", "dec", "dec_o"]\n'
    '    flagged = [f.strip() for f in features.split(",") if f.strip() in leakage]\n'
    '    safe = [f.strip() for f in features.split(",") if f.strip() not in leakage]\n'
    '    return json.dumps({"flagged_leakage": flagged, "safe_features": safe})\n\n'
    'def build_agent():\n'
    '    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)\n'
    '    return create_react_agent(llm, tools=[profile_data, check_leakage])\n\n'
    'if __name__ == "__main__":\n'
    '    agent = build_agent()\n'
    '    result = agent.invoke({"messages": [{"role": "user", "content": "Profile the dataset."}]})\n'
    '    print(result["messages"][-1].content)\n'
)

with open(f'{src_dir}\\agents.py', 'w') as f:
    f.write(agents_code)

print("src/agents.py saved successfully")

src/agents.py saved successfully


In [15]:
# Cell 12 — Final Summary
print("""
╔══════════════════════════════════════════════════════════╗
║        Love at First Agent — Project Summary             ║
╠══════════════════════════════════════════════════════════╣
║  Dataset:     Speed Dating (8,378 rows, 195 features)    ║
║  Match rate:  16.5% (class imbalanced)                   ║
║  Metric:      ROC AUC (recommended by agent)             ║
╠══════════════════════════════════════════════════════════╣
║  Pipeline:                                               ║
║  1. DataProfilerAgent  → stats + metric recommendation   ║
║  2. LeakageGuardAgent  → removes dec, dec_o              ║
║  3. ModelTrainerAgent  → XGBoost ROC AUC = 0.8553        ║
║  4. EvaluatorAgent     → live match probability          ║
╠══════════════════════════════════════════════════════════╣
║  Framework:   LangGraph ReAct + GPT-4o-mini              ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║        Love at First Agent — Project Summary             ║
╠══════════════════════════════════════════════════════════╣
║  Dataset:     Speed Dating (8,378 rows, 195 features)    ║
║  Match rate:  16.5% (class imbalanced)                   ║
║  Metric:      ROC AUC (recommended by agent)             ║
╠══════════════════════════════════════════════════════════╣
║  Pipeline:                                               ║
║  1. DataProfilerAgent  → stats + metric recommendation   ║
║  2. LeakageGuardAgent  → removes dec, dec_o              ║
║  3. ModelTrainerAgent  → XGBoost ROC AUC = 0.8553        ║
║  4. EvaluatorAgent     → live match probability          ║
╠══════════════════════════════════════════════════════════╣
║  Framework:   LangGraph ReAct + GPT-4o-mini              ║
╚══════════════════════════════════════════════════════════╝

